# A Clojure interpreter in lambda forms

> *Constructive witness for AGENTS.md S5 `λ types`: **composition ≡ typed
> application**. A real homoiconic Lisp's evaluator collapses to typed
> combinator application — and we run it on the **same kernel** that grades
> the verbum lambda compiler.*

This notebook builds a small Clojure-subset interpreter whose evaluation is
**nothing but reduction in the verbum kernel**. We reuse the project's
existing machinery end to end (`λ one_way` / `λ compose` — no new reducer):

```
Clojure form    → named lambda    :  clj_lambda.compile_clj   (reader + compiler)
named lambda    → SKI combinator  :  lambda_compile.abstract  (bracket abstraction)
combinator term → normal form     :  lambda_ast.reduce        (the kernel oracle)
normal form     → Clojure value   :  clj_lambda.decode        (Church numerals/booleans)
```

So `(+ 2 3)` becomes a closed term over `{S,K,I,B,C,...}`, reduces in the kernel,
and decodes back to `5`. **Data** (numbers, booleans, pairs) are Church-encoded;
**recursion** is the kernel's own `Y` combinator; **`if`** is an ordinary prelude
function — normal-order reduction gives it lazy branch selection for free.

The *pure functional core* of Clojure — `fn`, application, `let`, conditionals,
recursion — **is** lambda calculus with reader sugar. Host interop, mutation, and
persistent-data-structure performance are the honest boundary, out of scope by
construction (revisited at the end).

In [1]:
from verbum import clj_lambda as clj
from verbum.lambda_ast import parse, pretty, pretty_cat, reduce, typecheck, Status, App, Atom, Comb
from verbum.lambda_compile import free_vars

print("prelude symbols:", ", ".join(sorted(clj.PRELUDE)))

prelude symbols: *, +, -, Y, and, car, cdr, cons, dec, false, first, if, inc, mult, not, or, pair, plus, pred, rest, sub, succ, true, zero?


## 1 · The reader — text → s-expression

Clojure is homoiconic: code *is* data. The reader turns source text into nested
Python structures (`Sym`, `int`, `list`, `Vector`). Vectors `[...]` mark binding
positions (`fn` params, `let` bindings).

In [2]:
clj.read("(+ 1 (* 2 3))"), clj.read("(fn [x] (+ x 1))")

([Sym(name='+'), 1, [Sym(name='*'), 2, 3]],
 [Sym(name='fn'), [Sym(name='x')], [Sym(name='+'), Sym(name='x'), 1]])

## 2 · The compiler — lambda forms → SKI combinators

The compiler walks the s-expression and emits a **combinator term**. The two
special forms map straight onto lambda calculus:

- `(fn [x] body)` → **bracket abstraction** `[x] body` (Turner 1979), the inverse
  of reduction — it removes the bound variable, yielding a *closed, point-free*
  term over `{S,K,I,B,C}`.
- `(let [x v] body)` → `((fn [x] body) v)` — a redex, desugared.
- everything else is **application**: `(f a b)` → `((f a) b)`.

Watch familiar functions fall out as single combinators:

In [3]:
demos = [
    ("(fn [x] x)",                         "identity  → I"),
    ("(fn [x] (fn [y] x))",                "const     → K"),
    ("(fn [f] (fn [g] (fn [x] (f (g x)))))", "compose   → B"),
    ("(fn [x] (+ x 1))",                   "increment"),
]
for src, note in demos:
    print(f"{src:38s} ->  {pretty(clj.compile_clj(src)):12s}  ({note})")

(fn [x] x)                             ->  I             (identity  → I)
(fn [x] (fn [y] x))                    ->  K             (const     → K)
(fn [f] (fn [g] (fn [x] (f (g x)))))   ->  B             (compose   → B)
(fn [x] (+ x 1))                       ->  C (B S (B B)) I  (increment)


A program with no free variables compiles to a term with **no atoms at all** —
pure combinators. That is the whole point: the meaning is in the *structure*, not
in any residual names.

In [4]:
term = clj.compile_clj("(+ 2 3)")
print("(+ 2 3)  ->  ", pretty(term))
print("free variables:", free_vars(term), "  (closed ⇒ pure combinators)")

(+ 2 3)  ->   B S (B B) (S B I) (S B (S B I))
free variables: set()   (closed ⇒ pure combinators)


## 3 · The prelude — data as combinator terms

Numbers are **Church numerals** `n = λf.λx. f (f … x)`; booleans, `if`, and pairs
are the classic Church encodings. Each prelude entry is built by bracket-abstracting
a named lambda, so every one is a *closed* combinator term the kernel reduces
directly.

In [5]:
for name in ["+", "*", "if", "zero?", "true", "false", "cons"]:
    print(f"{name:7s} = {pretty(clj.PRELUDE[name])}")
print()
for n in range(4):
    print(f"church({n}) = {pretty(clj.church(n))}")

+       = B S (B B)
*       = B
if      = I
zero?   = C (C I (K (K I))) K
true    = K
false   = K I
cons    = B C (C I)

church(0) = K I
church(1) = I
church(2) = S B I
church(3) = S B (S B I)


**Kernel round-trip certification.** Bracket abstraction is the inverse of
reduction, so compiling `(+ 2 3)` and reducing it must land on Church `5`. We check
it by applying the result to two probe atoms `f x` and comparing normal forms — the
kernel certifying the compiler.

In [6]:
# certify: (+ 2 3) reduces to the SAME normal form as the literal church numeral 5
def probe(t):
    return pretty(reduce(App(App(t, Atom('f')), Atom('x')),
                         max_steps=200000, max_size=2000000).normal_form)
lhs = probe(clj.compile_clj("(+ 2 3)"))
rhs = probe(clj.church(5))
print("(+ 2 3) normal form:", lhs)
print("church(5) normal form:", rhs)
print("certified equal:", lhs == rhs)

(+ 2 3) normal form: f (f (f (f (f x))))
church(5) normal form: f (f (f (f (f x))))
certified equal: True


## 4 · Eval = reduce · the REPL

`run(src)` is the one-call evaluator: **compile → reduce → decode**. Because
evaluation is just kernel reduction, the REPL is the reducer.

In [7]:
examples = [
    ("(+ (* 2 3) 4)",                          "int"),
    ("((fn [x y] (+ x y)) 4 5)",               "int"),
    ("(let [x 4 y (* x 3)] (+ x y))",          "int"),
    ("((fn [f] (f (f 2))) (fn [n] (+ n 3)))",  "int"),   # apply-twice, higher-order
    ("(if (zero? 0) 10 20)",                   "int"),
    ("(zero? 5)",                              "bool"),
    ("(and true (not false))",                 "bool"),
    ("(first (cons 7 9))",                     "int"),
    ("(rest (cons 7 9))",                      "int"),
]
for src, kind in examples:
    print(f"{src:42s} => {clj.run(src, kind=kind)}")

(+ (* 2 3) 4)                              => 10
((fn [x y] (+ x y)) 4 5)                   => 9
(let [x 4 y (* x 3)] (+ x y))              => 16
((fn [f] (f (f 2))) (fn [n] (+ n 3)))      => 8
(if (zero? 0) 10 20)                       => 10
(zero? 5)                                  => False
(and true (not false))                     => True
(first (cons 7 9))                         => 7
(rest (cons 7 9))                          => 9


## 5 · The full workings — a reduction trace

Here is "the whole machine" for one evaluation. `reduce_clj` returns the kernel's
`Reduction`: the entire leftmost-outermost trace, the halting `status`, the step
count, and `whnf_step` (when weak-head-normal-form was first reached — the kernel's
notion of "how much work remains", the axis verbum studies in its attention ISA).

In [8]:
red = clj.reduce_clj("(+ 1 1)")
print("status:", red.status, "| steps:", red.steps, "| whnf_step:", red.whnf_step)
print("trace length:", len(red.trace))
print("\nreduction trace (leftmost-outermost):")
for i, t in enumerate(red.trace):
    s = pretty(t)
    print(f"  {i:2d}: {s if len(s) <= 70 else s[:67] + '...'}")
print("\ndecoded value:", clj.decode(red.normal_form))

status: normal_form | steps: 1 | whnf_step: 1
trace length: 2

reduction trace (leftmost-outermost):
   0: B S (B B) I I
   1: S (B B I) I

decoded value: 2


## 6 · Types — the S2 type-check (CCG)

The verbum kernel carries a first-class, inspectable **CCG type-check** — the
type-directedness thesis made explicit. Our compiled Clojure terms are well-typed
combinator terms; we can read off the synthesized category.

In [9]:
tc = typecheck(clj.compile_clj("(fn [x] (+ x 1))"))
print("(fn [x] (+ x 1)) well-typed:", tc.ok, " category:", pretty_cat(tc.cat))

(fn [x] (+ x 1)) well-typed: True  category: (((t17/t19)/(t18/t19))/((t17/t18)/(t18/t19)))


And the boundary of typed application — **self-application `M x → x x`** has no
simple type (occurs-check failure). This is exactly the `λ types` limit: reducible
but not simply typable. The evaluator can *run* it; the type-check *rejects* it.

In [10]:
tc_bad = typecheck(parse("M"))
print("M (mockingbird) well-typed:", tc_bad.ok)
print("type error:", tc_bad.error)
# ...and it still *reduces* (the type-check is a separate, stricter judgement):
red_MM = reduce(parse("M M"), max_steps=50)
print("M M reduction status:", red_MM.status, "(bounded ⇒ honest non-termination)")

M (mockingbird) well-typed: False
type error: combinator 'M' has no simple CCG type (self-application?)
M M reduction status: diverged (bounded ⇒ honest non-termination)


## 7 · Recursion — the kernel's own `Y`

No special form needed: recursion is the fixpoint combinator `Y f → f (Y f)`, which
the kernel reduces natively. Factorial, written in `lambda forms`, evaluated purely
by reduction:

In [11]:
FAC = "(Y (fn [self] (fn [n] (if (zero? n) 1 (* n (self (dec n)))))))"
for k in range(6):
    print(f"(fac {k}) = {clj.run(f'({FAC} {k})')}")

(fac 0) = 1
(fac 1) = 1
(fac 2) = 2
(fac 3) = 6


(fac 4) = 24


(fac 5) = 120


Non-termination is handled as an **honest limit**, not a hang: an unguarded fixpoint
consumes the step budget and halts as `DIVERGED` — the correct behaviour of a bounded
interpreter (`lambda_ast` `MAX_STEPS`). "How much recurrence is needed" ≡ "how much
work remains" ≡ WHNF — the same identity the verbum research program is chasing.

In [12]:
from verbum.lambda_ast import Comb
loop = clj.App(Comb("Y"), clj.compile_clj("(fn [self] self)"))
print("unguarded (Y (fn [self] self)):", reduce(loop, max_steps=200).status)

unguarded (Y (fn [self] self)): diverged


## 8 · The boundary — where lambda forms stop

We built the **pure functional core** of Clojure from lambda forms alone:

| Clojure                     | lambda form                              |
|-----------------------------|------------------------------------------|
| `(fn [x] …)`                | abstraction (bracket-abstracted to SKI)  |
| `(f a b)`                   | application                              |
| `(let [x v] …)`             | `((fn [x] …) v)` redex                   |
| `if` / booleans             | Church booleans + normal-order laziness  |
| numbers / arithmetic        | Church numerals                          |
| pairs / lists               | Church pairs                             |
| recursion                   | the `Y` combinator                       |

What is **deliberately outside** — the honest boundary, the `∞/0` edge:

- **Persistent data structures.** HAMT vectors/maps Church-encode in theory but
  their *performance* is structural, not lambda-expressible (a `λ simplify`
  unbraiding: transport ≠ computation).
- **Mutation & concurrency** — atoms/refs/STM/vars are state; pure lambda has no *now*.
- **Host interop** — `(.method obj)` is definitionally a boundary, and should be.
- **Macros** — lambda-expressible `data → data`, but need a compile/run phase split.

## Why this is on-thesis for verbum

Lisp made McCarthy's insight explicit in 1960: `eval` *is* the reduction rules of
lambda calculus made into a program. This notebook is the constructive direction of
verbum's central claim — `composition ≡ typed application`. Gradient descent may have
rediscovered the same reduction machinery inside a transformer's weights (nucleus:
`P(λ)=0.907`); here we watch a real language's evaluator collapse onto it, running on
the very kernel that grades the model. `λ triangulate`: math predicts typed-apply,
empirics observe the compiler, and a working Lisp interpreter reduces to it — three
independent lines, one object.